# Cálculo de métricas de classificação — DIO
**Autor:** Giordani Martins Silva. Código e documentação preparados com auxílio de IA.

Objetivo: implementar acurácia, sensibilidade (recall), especificidade, precisão e F-score pelas fórmulas. Usamos uma matriz binária arbitrária, como permite o desafio; não foi treinado um modelo nem utilizada uma base real.

Execute as células em ordem. Não requer GPU, downloads ou bibliotecas de aprendizado de máquina. O notebook usa somente a biblioteca padrão do Python.


## 1. Definir a classe positiva e a matriz
Exemplo fictício: **positivo = peça com defeito**, negativo = peça sem defeito.

| Real / Previsto | Negativo | Positivo |
| --- | --- | --- |
| Negativo | VN = 45 | FP = 10 |
| Positivo | FN = 5 | VP = 40 |

São 100 peças fictícias. VP e VN são acertos; FP são alarmes falsos; FN são defeitos não detectados. A ordem das linhas e colunas é sempre negativo, positivo. Não são dados de uma empresa.


## 2. Fórmulas
Com N = VP + VN + FP + FN:

| Métrica | Fórmula | Pergunta |
| --- | --- | --- |
| Acurácia | (VP + VN) / N | Qual fração de todas as previsões está correta? |
| Sensibilidade / recall | VP / (VP + FN) | Dos positivos reais, quantos foram detectados? |
| Especificidade | VN / (VN + FP) | Dos negativos reais, quantos foram reconhecidos? |
| Precisão | VP / (VP + FP) | Dos positivos previstos, quantos eram positivos reais? |
| F1-score | 2 × P × S / (P + S) | Como equilibrar precisão e sensibilidade? |

O F-score escolhido é F1, com pesos iguais. Pelo cálculo direto das contagens: F1 = 2VP / (2VP + FP + FN). Essa forma permite F1=0 quando há erro e VP=0, mesmo se uma das métricas intermediárias não for definida. Se o próprio denominador for zero, retornamos None, não zero.


In [ ]:
"""Métricas binárias implementadas pelas fórmulas, sem biblioteca de ML."""
import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class MatrizConfusao:
    vp: int
    vn: int
    fp: int
    fn: int

    def __post_init__(self):
        for name, value in asdict(self).items():
            if type(value) is not int or value < 0:
                raise ValueError(f"{name} deve ser inteiro não negativo.")

    @property
    def total(self):
        return self.vp + self.vn + self.fp + self.fn

    def como_matriz(self):
        """Linhas: real; colunas: previsto. Ordem: negativo, positivo."""
        return [[self.vn, self.fp], [self.fn, self.vp]]


def dividir(numerador, denominador):
    # None explicita uma métrica indefinida, sem fabricar um resultado zero.
    return numerador / denominador if denominador else None


def acuracia(matriz):
    return dividir(matriz.vp + matriz.vn, matriz.total)


def sensibilidade(matriz):
    return dividir(matriz.vp, matriz.vp + matriz.fn)


def especificidade(matriz):
    return dividir(matriz.vn, matriz.vn + matriz.fp)


def precisao(matriz):
    return dividir(matriz.vp, matriz.vp + matriz.fp)


def f_score(matriz):
    # F-score aqui é F1: média harmônica com pesos iguais para precisão e recall.
    # A forma por contagens evita intermediários indefinidos quando VP=0.
    return dividir(2 * matriz.vp, 2 * matriz.vp + matriz.fp + matriz.fn)


def calcular_metricas(matriz):
    return {"acuracia": acuracia(matriz), "sensibilidade_recall": sensibilidade(matriz),
            "especificidade": especificidade(matriz), "precisao": precisao(matriz),
            "f1_score": f_score(matriz)}


def percentual(valor):
    return "indefinida (denominador zero)" if valor is None else f"{100 * valor:.2f}%"


def gerar_relatorio(matriz, output_dir="resultados"):
    metrics = calcular_metricas(matriz)
    result = {"tipo": "classificação binária", "contagens": asdict(matriz),
              "total": matriz.total, "matriz": matriz.como_matriz(),
              "ordem": {"linhas": "real: negativo, positivo", "colunas": "previsto: negativo, positivo"},
              "metricas": metrics,
              "divisao_por_zero": "None no Python; null no JSON; indefinida no relatório"}
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    (output / "metricas.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    table = "\n".join(f"| {name} | {percentual(value)} |" for name, value in metrics.items())
    text = f"""# Avaliação da matriz de confusão

Total: {matriz.total}. Classe positiva e origem dos dados devem ser definidas pelo experimento.

| Real / Previsto | Negativo | Positivo |
| --- | --- | --- |
| Negativo | VN = {matriz.vn} | FP = {matriz.fp} |
| Positivo | FN = {matriz.fn} | VP = {matriz.vp} |

| Métrica | Resultado |
| --- | --- |
{table}

Os valores do JSON mantêm a precisão dos cálculos; a exibição é arredondada para duas casas percentuais.
Uma métrica indefinida não equivale a zero. F-score representa F1 neste projeto.
"""
    (output / "relatorio.md").write_text(text, encoding="utf-8")
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--vp", type=int, default=40)
    parser.add_argument("--vn", type=int, default=45)
    parser.add_argument("--fp", type=int, default=10)
    parser.add_argument("--fn", type=int, default=5)
    parser.add_argument("--saida", default="resultados")
    args = parser.parse_args()
    try:
        matrix = MatrizConfusao(args.vp, args.vn, args.fp, args.fn)
        result = gerar_relatorio(matrix, args.saida)
    except (ValueError, OSError) as error:
        parser.exit(1, f"Erro: {error}\n")
    print("Matriz: linhas reais e colunas previstas; ordem negativo, positivo")
    print(matrix.como_matriz())
    print(f"Total: {matrix.total}")
    for name, value in result["metricas"].items():
        print(f"{name}: {percentual(value)}")



## 3. Calcular e exportar
Altere as quatro contagens abaixo para testar outra matriz. Contagens negativas ou não inteiras são rejeitadas.

In [ ]:
matriz = MatrizConfusao(vp=40, vn=45, fp=10, fn=5)
resultado = gerar_relatorio(matriz, "resultados")
print("Matriz (real nas linhas; previsto nas colunas):", matriz.como_matriz())
print("Total:", matriz.total)
for nome, valor in resultado["metricas"].items():
    print(f"{nome}: {percentual(valor)}")


## 4. Conferir um cálculo manual
Para a matriz padrão: 85/100 = 85% de acurácia; 40/45 de sensibilidade; 45/55 de especificidade; 40/50 de precisão; 80/95 de F1. A checagem abaixo usa uma matriz fixa própria, sem impedir que você altere a anterior.

In [ ]:
from math import isclose
exemplo = calcular_metricas(MatrizConfusao(40, 45, 10, 5))
esperado = {"acuracia": 85/100, "sensibilidade_recall": 40/45,
            "especificidade": 45/55, "precisao": 40/50, "f1_score": 80/95}
assert all(isclose(exemplo[nome], valor) for nome, valor in esperado.items())
print("Cinco cálculos conferidos.")


## 5. Entender denominadores zero
Sem positivos previstos, precisão é indefinida. Sem positivos reais, recall é indefinido. Sem negativos reais, especificidade é indefinida. Matriz vazia não permite calcular acurácia. None vira null no JSON.

In [ ]:
casos = {
    "matriz vazia": MatrizConfusao(0, 0, 0, 0),
    "sem positivos previstos": MatrizConfusao(0, 10, 0, 5),
    "sem positivos reais": MatrizConfusao(0, 10, 5, 0),
}
for nome, caso in casos.items():
    print(nome, calcular_metricas(caso))


## 6. Consultar os arquivos gerados
O JSON preserva os valores decimais calculados, e o Markdown exibe percentuais com duas casas. No Colab, os arquivos estão no painel lateral da pasta resultados.

In [ ]:
print(Path("resultados/relatorio.md").read_text(encoding="utf-8"))
print("JSON:", Path("resultados/metricas.json").resolve())


## Reflexão
- Precisão e sensibilidade têm denominadores diferentes: confundi-las muda a interpretação.
- Muitos FN reduzem o recall; muitos FP reduzem a precisão e a especificidade.
- Acurácia elevada pode esconder falhas na classe rara. Analise a matriz e várias métricas.
- F1 não usa VN diretamente; a classe positiva escolhida deve estar explícita.
- Este exercício calcula métricas de uma matriz arbitrária, não comprova desempenho de um modelo real.

O notebook é publicado sem saídas para execução limpa. As evidências locais estão no repositório; não foi realizada uma sessão no Colab durante a preparação.
